In [1]:
import torch
import pyarrow.parquet as pq
from torch.utils.data import DataLoader,Dataset
import pandas as pd
import numpy as np


In [2]:
class ToTensor:
    def __call__(self, features, target):
        features = torch.tensor(features, dtype=torch.float32)
        target = torch.tensor(target, dtype=torch.float32)
        return features, target


class CMAPSSDataset(Dataset):

    def __init__(self, parquet_path, window_size=30, transform=None):

        df = pd.read_parquet(parquet_path)

        self.window_size = window_size
        self.transform = transform

        self.global_features = []
        self.global_targets = []

        feature_columns = [
            col for col in df.columns
            if col not in ["time_cycle", "RUL", "global_engine_id"]
        ]

        for _, engine_df in df.groupby("global_engine_id"):

            engine_features = engine_df[feature_columns].astype(np.float32).values
            engine_targets = engine_df["RUL"].astype(np.float32).values

            num_cycles = len(engine_df)
            num_windows = num_cycles - window_size + 1

            if num_windows <= 0:
                continue

            for i in range(num_windows):

                self.global_features.append(
                    engine_features[i:i + window_size]
                )

                self.global_targets.append(
                    engine_targets[i + window_size - 1]
                )

        self.global_features = np.asarray(
            self.global_features,
            dtype=np.float32
        )

        self.global_targets = np.asarray(
            self.global_targets,
            dtype=np.float32
        )

    def __len__(self):
        return len(self.global_features)

    def __getitem__(self, idx):

        features = self.global_features[idx]
        target = self.global_targets[idx]

        if self.transform is not None:
            features, target = self.transform(features, target)

        return features, target

In [6]:
import torch.nn as nn
class CMAPSSModel(nn.Module):
    def __init__(self,input_size = 24,hidden_size=128,num_layers =3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )
        self.fc1 = nn.Linear(
            in_features=hidden_size,
            out_features=1
        )
        
    def forward(self,x):
        out,(h_n,c_n) = self.lstm(x)
        out =  out[:,-1,:]
        out = self.fc1(out)
        return out
    
if __name__=="__main__":
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    parquet_path = r"C:\projects_data\ml_pipeline\data\output_parquet.parquet"
    json_path = r"C:\projects_data\ml_pipeline\data\output_json.json"
    dataset = CMAPSSDataset(parquet_path = parquet_path,window_size=30)
    dataloader = DataLoader(
        dataset,
        batch_size=64,
        shuffle=True,
        num_workers=0,
        drop_last=True
    
    )
    lstm = CMAPSSModel().to(device)
    optim = torch.optim.AdamW(lstm.parameters(), lr=0.0001)
    epochs = 5
    criterion = nn.MSELoss()
    lstm.train()
    for epoch in range(epochs):
        lstm.train()
        running_loss = 0.0
        for x,y in dataloader:
            x = x.to(device)
            y = y.to(device)
            optim.zero_grad()
            predictions = lstm(x)
            y = y.unsqueeze(1)
            loss = criterion(predictions,y)
            loss.backward()
            optim.step()
            running_loss+=loss.item()
        epoch_loss = running_loss/len(dataloader)
        print(
            f"Epoch : [{epoch+1}/{epochs}]"
            f"Loss: {epoch_loss:.4f}"
        )
            
            
        
        

KeyboardInterrupt: 

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


cpu
